# Jow @ EXIST 2026 — Task 2 (Memes): camada de confusão como PERDA AUXILIAR (Task A)

Cópia de `memes_model.ipynb` com **um único acrescento**: o termo auxiliar

    L = L_soft(A, B, C) + lambda * L_aux(A)

`L_soft` é a perda do A0, **inalterada**. `L_aux` modela o voto de cada anotador
como `p_voto(a, i) = p_latente(i) @ C_a`, com `C_a` [2x2] de linhas normalizadas.

Linha de **caracterização**, não de melhoria: um resultado nulo é válido e esperado.

Protocolo, grelha e critério pré-registado: `reliability/CONFUSION_EXPERIMENT.md`.
Implementação e testes: `EXIST_MEMES/confusion_core.py`, `EXIST_MEMES/confusion_selftest.py`.

**Ordem de execução obrigatória:** primeiro o controlo `METHOD='M3'`, `LAMBDA=0.0`,
`SEED=42` — tem de reproduzir `raw_seed42` (A0). Só depois a grelha do M1.


**TESTE DE 2 PASSOS — não é uma run do estudo.** `max_steps=2`, sem avaliação e
sem checkpoints. Serve só para comparar a `loss` do passo 1 entre este notebook e
o par correspondente (`memes_model_step2` vs `memes_confusion_step2`), para saber
se a divergência do invólucro nasce no forward ou no lado do optimizador.

## Glossário de conceitos usados neste notebook



| Termo | O que significa aqui |

|---|---|

| **Token / tokenizar** | Cortar o texto em pedacinhos (palavras ou sub-palavras) que o modelo consegue processar como números. |

| **CLS token** | Um token especial adicionado no início de todo texto/imagem processado por um Transformer. Depois de passar pelo modelo, o vetor desse token resume "o significado geral" daquele texto ou imagem inteira — é o que usamos como representação da entrada. |

| **Embedding / vetor** | Uma lista de números que representa algo (uma palavra, uma imagem, uma pessoa) de um jeito que o modelo consegue comparar matematicamente. |

| **Soft label (label suave)** | Em vez de um rótulo único ("YES"), uma distribuição de probabilidade que reflete o desacordo entre anotadores, ex. `[0.67, 0.33]` para `[YES, NO]`. |

| **Hard label (label duro)** | O rótulo único que se obtém "arredondando" o soft label (ex.: pega a opção com maior probabilidade, ou aplica um corte/threshold). |

| **Gate (portão)** | Um número entre 0 e 1 (calculado pelo próprio modelo via `sigmoid`) que decide "quanto" de um sinal deixar passar. Funciona como um botão de volume que o modelo aprende a girar sozinho. |

| **LayerNorm** | Uma operação que reescala os números de um vetor para que fiquem numa faixa estável — evita que alguns números "explodam" e dominem o treino. |

| **Dropout** | Durante o treino, desliga aleatoriamente uma fração dos neurônios a cada passo — força o modelo a não depender demais de um único caminho, reduzindo overfitting. |

| **BCEWithLogitsLoss** | Função de erro usada quando a resposta é binária (sim/não) ou múltiplas respostas independentes (Task C, onde mais de uma categoria pode estar certa ao mesmo tempo). |

| **Softmax + cross-entropy** | Função de erro usada quando as opções são mutuamente exclusivas (Task B: só uma intenção por vez). |

| **Fine-tuning parcial / congelar camadas** | "Congelar" uma parte do modelo pré-treinado significa não deixá-la mudar durante o treino (economiza memória e evita destruir o que ela já sabe); só as últimas camadas ficam "descongeladas" (treináveis). |

| **ICM-Norm (soft / hard)** | Métrica oficial da competição EXIST, calculada pela biblioteca PyEvALL. Mede o quão perto a previsão do modelo está do gabarito — a versão *soft* compara distribuições inteiras, a versão *hard* compara rótulos únicos. |

| **Threshold (limiar)** | Valor de corte usado para transformar uma probabilidade em decisão binária, ex.: "se P(categoria) ≥ 0.35, marca essa categoria como presente". |



**Nota sobre versões:** este notebook roda na VM de treino do projeto — `transformers==5.6.0`,

`torch==2.6.0+cu124` (ver `CLAUDE.md` do repositório para a lista completa de versões).


## 1. Instalação de dependências

In [1]:
%pip install transformers
%pip install accelerate
%pip install emoji wordsegment
%pip install huggingface_hub
%pip install torch --index-url https://download.pytorch.org/whl/cu124
%pip install pandas scikit-learn
%pip install ftfy regex tqdm
%pip install PyEvALL

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 2. Imports

In [2]:
import os
import re
import json
import csv
import tempfile
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image
import emoji
from wordsegment import load as load_wordsegment_dictionary, segment

from sklearn.model_selection import train_test_split

from transformers import (
    XLMRobertaTokenizer, XLMRobertaModel,
    CLIPProcessor, CLIPVisionModel,
    TrainingArguments, Trainer, TrainerCallback,
)

from pyevall.evaluation import PyEvALLEvaluation
from pyevall.utils.utils import PyEvALLUtils

os.chdir("/home/a67606/Thesis/EXIST_MEMES")
load_wordsegment_dictionary()

from confusion_core import (
    build_vote_table,
    VoteAugmentedDataset,
    ConfusionMultiHeadModel,
    make_confusion_trainer_class,
    make_confusion_logger_class,
)


In [3]:
import hashlib
import random

# ======================= CONFIGURAÇÃO DA RUN =======================
SEED   = 42
METHOD = "M1"      # "M1" (Chu, comum+individual) | "M2" (Tanno, traço) | "M3" (crowd layer)
LAMBDA = 0.0       # 0.0 = CONTROLO (tem de reproduzir o A0); grelha: 0.1, 0.3, 1.0
                   # O controlo corre com METHOD="M1": o `common` = 3*I é um detector
                   # vivo de weight decay; o `delta` do M3 nasce a 0 e o decay nele
                   # seria invisível (0 -> 0).
REG_WEIGHT = None  # None = valor a priori do método (M1 0.01, M2 0.01, M3 0.0)

ARM = "raw"        # alvo de treino da Task A: SEMPRE o A0 (soft label crua)
SOFT_A_PATH = f'../reliability/outputs/soft_labels_taskA_{ARM}.json'
RUN_TAG = f'conf_{METHOD}_lam{LAMBDA}_seed{SEED}'
# ===================================================================

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


def set_full_determinism(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)


def model_fingerprint(model):
    digest = hashlib.sha256()
    for name, param in sorted(model.state_dict().items()):
        digest.update(name.encode())
        digest.update(param.detach().cpu().numpy().tobytes())
    return digest.hexdigest()[:16]


set_full_determinism(SEED)
print(f"RUN_TAG = {RUN_TAG}")


RUN_TAG = conf_M1_lam0.0_seed42


## 3. Carregar os dados



O dataset (`EXIST2026_training.json`) vem como um dicionário `{id_do_meme: {...campos...}}`.

`pd.read_json(...).transpose()` transforma isso numa tabela (DataFrame) com uma linha por meme.


In [4]:
raw_data = pd.read_json('./Dataset/memes/training/EXIST2026_training.json')
raw_data = raw_data.transpose()
raw_data.head()

,id_EXIST,lang,text,meme,path_memes,number_annotators,annotators,gender_annotators,age_annotators,ethnicities_annotators,study_levels_annotators,countries_annotators,sensorial,labels_task2_1,labels_task2_2,labels_task2_3,split
110887,110887,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,memes/110887.jpeg,6,"[Annotator_193, Annotator_194, Annotator_195, ...","[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Multiracial, White or Cau...","[High school degree or equivalent, Master’s de...","[Sweden, United States, Spain, Portugal, Spain...","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, YES, YES, YES, YES, YES]","[JUDGEMENTAL, DIRECT, DIRECT, DIRECT, DIRECT, ...","[[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION], [M...",TRAIN-MEME_ES
110466,110466,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,memes/110466.jpeg,6,"[Annotator_103, Annotator_104, Annotator_105, ...","[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, White or Caucasian, Hispa...","[Bachelor’s degree, High school degree or equi...","[Spain, Portugal, Spain, Canada, United States...","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, NO, YES, YES, NO, NO]","[DIRECT, -, DIRECT, DIRECT, -, -]","[[STEREOTYPING-DOMINANCE], [-], [STEREOTYPING-...",TRAIN-MEME_ES
111269,111269,es,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,memes/111269.jpeg,6,"[Annotator_277, Annotator_278, Annotator_279, ...","[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Hispano or Latino, other,...","[Bachelor’s degree, Master’s degree, High scho...","[Spain, Spain, Spain, Italy, Spain, Portugal]","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, YES, YES, YES, YES, NO]","[DIRECT, DIRECT, DIRECT, DIRECT, DIRECT, -]","[[OBJECTIFICATION, SEXUAL-VIOLENCE], [OBJECTIF...",TRAIN-MEME_ES
110593,110593,es,HOY QUIERO FELICITAR A TODAS LAS MUJERES DE ES...,110593.jpeg,memes/110593.jpeg,6,"[Annotator_127, Annotator_128, Annotator_129, ...","[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Hispano or Latino, White ...","[High school degree or equivalent, Master’s de...","[Poland, Portugal, Spain, United Kingdom, Mexi...","{'users': ['ES1', 'ES3'], 'modalities': {'ET':...","[NO, YES, NO, NO, NO, NO]","[-, JUDGEMENTAL, -, -, -, -]","[[-], [STEREOTYPING-DOMINANCE, OBJECTIFICATION...",TRAIN-MEME_ES
110946,110946,es,DUCHATE GUARRA GRACIAS memegenerator.es,110946.jpeg,memes/110946.jpeg,6,"[Annotator_211, Annotator_212, Annotator_213, ...","[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Hispano or Latino, White ...","[Bachelor’s degree, Bachelor’s degree, Bachelo...","[Portugal, Chile, Italy, Portugal, Spain, Chile]","{'users': ['ES1', 'ES2', 'ES3', 'ES4'], 'modal...","[YES, YES, YES, YES, NO, NO]","[DIRECT, DIRECT, DIRECT, DIRECT, -, -]","[[OBJECTIFICATION], [SEXUAL-VIOLENCE], [OBJECT...",TRAIN-MEME_ES


In [5]:
def has_biometric_modality(row, modality_name):
    sensorial_data = row.get('sensorial')
    if sensorial_data is None or not isinstance(sensorial_data, dict):
        return False
    users_with_data = sensorial_data.get('modalities', {}).get(modality_name, {}).get('by_user', {})
    return len(users_with_data) > 0

print(f"Total de memes: {len(raw_data)}")
print(f"Com algum dado biométrico: {raw_data.apply(lambda r: r.get('sensorial') is not None, axis=1).sum()}")
for modality in ['ET', 'HR', 'EEG']:
    count = raw_data.apply(lambda r, modality=modality: has_biometric_modality(r, modality), axis=1).sum()
    print(f"Com {modality}: {count}")

Total de memes: 3984
Com algum dado biométrico: 3984
Com ET: 3984
Com HR: 3984
Com EEG: 3984


## 4. Remover colunas que não usamos



`id_EXIST`, `number_annotators`, `annotators`, `lang`, `split` são metadados administrativos

do dataset — não entram no modelo.


In [6]:
dataset = raw_data.copy()
dataset.drop(['id_EXIST', 'number_annotators', 'annotators', 'lang', 'split'], axis=1, inplace=True)
dataset.describe()

,text,meme,path_memes,gender_annotators,age_annotators,ethnicities_annotators,study_levels_annotators,countries_annotators,sensorial,labels_task2_1,labels_task2_2,labels_task2_3
count,3984,3984,3984,3984,3984,3984,3984,3984,3984,3984,3984,3984
unique,3981,3984,3984,1,2,111,125,151,3984,64,785,2889
top,"Para las mujeres: ""¡Hola hermosa!"" Halago Para...",110887.jpeg,memes/110887.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 18-22, 23-45, 46+]","[White or Caucasian, White or Caucasian, White...","[High school degree or equivalent, Bachelor’s ...","[Sweden, United States, Spain, Portugal, Spain...","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, YES, YES, YES, YES, YES]","[-, -, -, -, -, -]","[[-], [-], [-], [-], [-], [-]]"
freq,2,1,1,3984,2005,161,81,27,1,572,373,373


In [7]:
for i in range(3):
    path = os.path.join('./Dataset/memes/training/', dataset['path_memes'].iloc[i])
    exists = os.path.exists(path)
    print(f"{path} -> {'existe' if exists else 'FALTANDO'}")

./Dataset/memes/training/memes/110887.jpeg -> existe
./Dataset/memes/training/memes/110466.jpeg -> existe
./Dataset/memes/training/memes/111269.jpeg -> existe


## 5. Definições de labels (rótulos)



Cada subtask tem sua própria lista fixa de rótulos possíveis. `THRESHOLD_GRID` é a lista de

valores de corte que testamos mais adiante para decidir "esse limiar de probabilidade conta

como presente ou não" (Task C). As hierarquias (`TASK_2_2_HIERARCHY`, `TASK_2_3_HIERARCHY`)

dizem à métrica oficial (PyEvALL) que, por exemplo, errar "DIRECT" por "JUDGEMENTAL" é um erro

menor do que errar "sexista" por "não-sexista" — ambos são sub-categorias de "YES".



⚠️ Estas definições existem **uma única vez** neste notebook. No notebook original elas

apareciam duplicadas em duas células — inofensivo porque os valores batiam, mas é um padrão

arriscado (se alguém editasse só uma cópia, os índices dessincronizariam silenciosamente).


In [8]:
LABELS_2_1 = ['YES', 'NO']
LABELS_2_2 = ['NONE', 'JUDGEMENTAL', 'DIRECT']
LABELS_2_3 = ['NONE', 'IDEOLOGICAL-INEQUALITY', 'MISOGYNY-NON-SEXUAL-VIOLENCE',
              'SEXUAL-VIOLENCE', 'OBJECTIFICATION', 'STEREOTYPING-DOMINANCE']

THRESHOLD_GRID = np.arange(0.05, 0.65, 0.05).tolist()

TASK_2_2_HIERARCHY = {
    "YES": ["DIRECT", "JUDGEMENTAL"],
    "NO": []
}

TASK_2_3_HIERARCHY = {
    "YES": [
        "IDEOLOGICAL-INEQUALITY",
        "STEREOTYPING-DOMINANCE",
        "OBJECTIFICATION",
        "SEXUAL-VIOLENCE",
        "MISOGYNY-NON-SEXUAL-VIOLENCE"
    ],
    "NO": []
}

In [9]:
def normalize_hashtags(text):
    def split_one_hashtag(match):
        tag_without_hash = match.group()[1:]
        if "_" in tag_without_hash:
            words = tag_without_hash.split("_")
        elif re.search(r'[A-Z]', tag_without_hash):
            words = re.sub(r'(?<!^)(?=[A-Z])', ' ', tag_without_hash).split()
        else:
            try:
                words = segment(tag_without_hash)
            except ValueError:
                words = [tag_without_hash]
        return " ".join(words)
    return re.sub(r'#\w+', split_one_hashtag, text)

In [10]:
def convert_emojis_to_text(text):
    return emoji.demojize(text)

In [11]:
def replace_urls(text):
    return re.sub(r'http\S+|www\S+', '<URL>', text)

In [12]:
def replace_user_mentions(text):
    return re.sub(r'@\w+', '<USER_MENTION>', text)

In [13]:
def replace_profanity(text):
    profanity_pattern = r'\b[a-zA-Z]*[*!@#$%^&]+[a-zA-Z]*\b'
    return re.sub(profanity_pattern, "<PROFANITY>", text)

In [14]:
def limit_token_repetition(text, max_repeats=2):
    words = text.split()
    kept_words = []
    previous_word = None
    repeat_count = 0
    for word in words:
        if word == previous_word:
            repeat_count += 1
        else:
            previous_word = word
            repeat_count = 1
        if repeat_count <= max_repeats:
            kept_words.append(word)
    return " ".join(kept_words)

In [15]:
def strip_punctuation_and_extra_spaces(text):
    text = re.sub(r'[^\w\s<>]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

In [16]:
def pre_process_text(text):
    text = normalize_hashtags(text)
    text = text.lower()
    text = replace_urls(text)
    text = convert_emojis_to_text(text)
    text = replace_user_mentions(text)
    text = replace_profanity(text)
    text = limit_token_repetition(text)
    text = strip_punctuation_and_extra_spaces(text)
    return text

### 6.1 Exemplo de ponta a ponta



Para deixar claro o que o pipeline faz na prática, um exemplo real do dataset, antes e depois:


In [17]:
example_raw = dataset['text'].iloc[0]
example_clean = pre_process_text(example_raw)
print("ANTES:")
print(example_raw)
print()
print("DEPOIS:")
print(example_clean)

ANTES:
A VECES QUISIERAIR/ALZUMBA www.facebook.com/OiesamamadaOficial www.facebook.com/Ofesamamadaoficial 130 PEROVEOQUE LAS QUE VAN SIGUEN IGUAL DE GORDAS-Y-SE ME PASA 

DEPOIS:
a veces quisierair alzumba <URL> <URL> 130 peroveoque las que van siguen igual de gordas y se me pasa


In [18]:
def remap_labels_flat(annotator_labels):
    return [
        "NONE" if label == "-" else label
        for label in annotator_labels
        if label != "UNKNOWN"
    ]

def remap_labels_nested(per_annotator_label_lists):
    cleaned_lists = []
    for this_annotators_labels in per_annotator_label_lists:
        cleaned = [
            "NONE" if label == "-" else label
            for label in this_annotators_labels
            if label != "UNKNOWN"
        ]
        if cleaned:
            cleaned_lists.append(cleaned)
    return cleaned_lists

In [19]:
def get_soft_distribution(annotator_labels, all_possible_labels):
    if not annotator_labels:
        return None
    vote_counts = dict.fromkeys(all_possible_labels, 0)
    for vote in annotator_labels:
        if vote in vote_counts:
            vote_counts[vote] += 1
    num_votes = len(annotator_labels)
    return [vote_counts[label] / num_votes for label in all_possible_labels]

def annotations_matrix_to_distribution(per_annotator_label_lists, all_possible_labels):
    if not per_annotator_label_lists:
        return None
    label_counts = dict.fromkeys(all_possible_labels, 0)
    num_annotators = len(per_annotator_label_lists)
    for this_annotators_labels in per_annotator_label_lists:
        for label in this_annotators_labels:
            if label in label_counts:
                label_counts[label] += 1
    return [label_counts[label] / num_annotators for label in all_possible_labels]

## 8. Vetor demográfico dos anotadores



A ideia (inspirada na abordagem "GrootWatch"): se soubermos o perfil demográfico de quem

anotou cada meme (etnia, escolaridade, país), o modelo pode aprender que certos grupos

tendem a discordar mais em certos tipos de conteúdo — e usar isso como sinal extra.



**Como o vetor é montado, passo a passo:**

1. Para cada uma das 3 características (etnia / escolaridade / país), contamos quantos dos

   ~6 anotadores daquele meme caem em cada categoria possível.

2. Concatenamos as 3 listas de contagem numa única lista de 66 números (7 etnias + 6

   escolaridades + 53 países).

3. Dividimos a lista inteira pela soma total — assim cada posição vira "fração dos votos",

   não uma contagem bruta (isso é importante: normalizar dividindo pela soma preserva o

   sinal; testamos normalizar por norma L2 e o sinal demográfico foi destruído).



✅ **Corrigido nesta revisão (2026-08-01):** os dados brutos usam apóstrofo curvo (`’`,

ex. `"Bachelor’s degree"`), mas o vocabulário abaixo usava apóstrofo reto (`'`). Isso fazia

com que **64,6% dos votos de escolaridade nunca batessem com o vocabulário** e fossem

descartados silenciosamente do vetor (a fatia de escolaridade ficava quase sempre zerada).

`_normalize_apostrophes` (na função abaixo) trata os dois caracteres como equivalentes na

hora da comparação — o vocabulário e a ordem das 66 posições continuam exatamente os mesmos,

só a lógica de correspondência mudou. Efeito: **3.957 das 3.984 amostras têm o vetor

demográfico alterado** em relação ao sistema submetido (a normalização por soma agora usa o

denominador correto).


In [20]:
DEMO_ETHNICITY_VALS = sorted(['Asian', 'Black or African American', 'Hispano or Latino',
                               'Middle Eastern', 'Multiracial', 'White or Caucasian', 'other'])
DEMO_STUDY_VALS = sorted(["Bachelor's degree", 'Doctorate',
                           'High school degree or equivalent',
                           'Less than high school diploma', "Master's degree", 'other'])
DEMO_COUNTRY_VALS = sorted(['Afghanistan', 'Argentina', 'Australia', 'Austria', 'Azerbaijan',
                             'Bangladesh', 'Belgium', 'Brazil', 'Bulgaria', 'Canada', 'Chile',
                             'China', 'Colombia', 'Croatia', 'Cuba', 'Czech Republic',
                             'Dominican Republic', 'Ecuador', 'Estonia', 'Finland', 'France',
                             'Germany', 'Greece', 'Hungary', 'India', 'Ireland', 'Israel',
                             'Italy', 'Latvia', 'Mexico', 'Morocco', 'Netherlands',
                             'New Zealand', 'Nicaragua', 'Nigeria', 'Peru', 'Poland',
                             'Portugal', 'Romania', 'Russian Federation', 'Serbia', 'Slovenia',
                             'South Africa', 'Spain', 'Sweden', 'Turkey', 'United Arab Emirates',
                             'United Kingdom', 'United States', 'Uruguay', 'Venezuela',
                             'Viet Nam', 'Zimbabwe'])

DEMO_FEATURE_GROUPS = [
    ('ethnicities_annotators',  DEMO_ETHNICITY_VALS),
    ('study_levels_annotators', DEMO_STUDY_VALS),
    ('countries_annotators',    DEMO_COUNTRY_VALS),
]

DEMO_VEC_DIM = len(DEMO_ETHNICITY_VALS) + len(DEMO_STUDY_VALS) + len(DEMO_COUNTRY_VALS)
print(f"Dimensão do vetor demográfico: {DEMO_VEC_DIM}")

Dimensão do vetor demográfico: 66


In [21]:
def _normalize_apostrophes(text):
    return text.replace("’", "'")


def _count_votes_per_category(annotator_values, vocabulary):
    category_to_index = {
        _normalize_apostrophes(category): i for i, category in enumerate(vocabulary)
    }
    counts = np.zeros(len(vocabulary), dtype=np.float32)
    if not isinstance(annotator_values, list):
        return counts
    for value in annotator_values:
        if value is None:
            continue
        idx = category_to_index.get(_normalize_apostrophes(str(value).strip()), None)
        if idx is not None:
            counts[idx] += 1.0
    return counts

def compute_annotator_vector(row, feature_groups):
    category_count_vectors = [
        _count_votes_per_category(row.get(column_name, None), vocabulary)
        for column_name, vocabulary in feature_groups
    ]

    full_vector = np.concatenate(category_count_vectors)
    total_votes = full_vector.sum()
    if total_votes > 1e-9:
        full_vector = full_vector / total_votes
    return full_vector

## 9. Aplicar tudo ao dataset



Agora rodamos o pré-processamento de texto, o remapeamento + soft labels de cada task, e o

vetor demográfico, em todas as linhas do dataset. Por fim, removemos as poucas amostras sem

nenhum voto válido na Task A (a task principal — sem ela, não há o que treinar).


In [22]:
dataset['text_preprocessed'] = dataset['text'].apply(pre_process_text)

with open(SOFT_A_PATH) as f:
    _SOFT_A = json.load(f)
dataset['labels_task2_1_soft'] = dataset.index.map(lambda i: _SOFT_A[str(i)])
dataset['_id_frozen'] = dataset.index.astype(str)

dataset['labels_task2_2'] = dataset['labels_task2_2'].apply(remap_labels_flat)
dataset['labels_task2_2_soft'] = dataset['labels_task2_2'].apply(
    lambda votes: get_soft_distribution(votes, LABELS_2_2)
)

dataset['labels_task2_3'] = dataset['labels_task2_3'].apply(remap_labels_nested)
dataset['labels_task2_3_soft'] = dataset['labels_task2_3'].apply(
    lambda votes: annotations_matrix_to_distribution(votes, LABELS_2_3)
)

dataset['demo_vec'] = dataset.apply(
    lambda row: compute_annotator_vector(row, DEMO_FEATURE_GROUPS), axis=1
)

print(f"Antes de filtrar: {len(dataset)} amostras")
dataset = dataset[dataset['labels_task2_1_soft'].notna()].reset_index(drop=True)
print(f"Depois de filtrar (Task A precisa de ao menos 1 voto válido): {len(dataset)} amostras")
dataset.head()

Antes de filtrar: 3984 amostras
Depois de filtrar (Task A precisa de ao menos 1 voto válido): 3984 amostras


,text,meme,path_memes,gender_annotators,age_annotators,ethnicities_annotators,study_levels_annotators,countries_annotators,sensorial,labels_task2_1,labels_task2_2,labels_task2_3,text_preprocessed,labels_task2_1_soft,_id_frozen,labels_task2_2_soft,labels_task2_3_soft,demo_vec
0,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,memes/110887.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Multiracial, White or Cau...","[High school degree or equivalent, Master’s de...","[Sweden, United States, Spain, Portugal, Spain...","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, YES, YES, YES, YES, YES]","[JUDGEMENTAL, DIRECT, DIRECT, DIRECT, DIRECT, ...","[[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION], [M...",a veces quisierair alzumba <URL> <URL> 130 per...,"[1.0, 0.0]",110887,"[0.0, 0.16666666666666666, 0.8333333333333334]","[0.0, 0.16666666666666666, 0.5, 0.0, 0.6666666...","[0.0, 0.0, 0.11111111, 0.0, 0.055555556, 0.166..."
1,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,memes/110466.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, White or Caucasian, Hispa...","[Bachelor’s degree, High school degree or equi...","[Spain, Portugal, Spain, Canada, United States...","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, NO, YES, YES, NO, NO]","[DIRECT, NONE, DIRECT, DIRECT, NONE, NONE]","[[STEREOTYPING-DOMINANCE], [NONE], [STEREOTYPI...",se necesita cuidadora para adulto mayor fotos ...,"[0.5, 0.5]",110466,"[0.5, 0.0, 0.5]","[0.5, 0.16666666666666666, 0.16666666666666666...","[0.0, 0.0, 0.16666667, 0.0, 0.0, 0.16666667, 0..."
2,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,memes/111269.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Hispano or Latino, other,...","[Bachelor’s degree, Master’s degree, High scho...","[Spain, Spain, Spain, Italy, Spain, Portugal]","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, YES, YES, YES, YES, NO]","[DIRECT, DIRECT, DIRECT, DIRECT, DIRECT, NONE]","[[OBJECTIFICATION, SEXUAL-VIOLENCE], [OBJECTIF...",tomboy como son el anime y manga pre to tomboy...,"[0.8333333333333334, 0.16666666666666666]",111269,"[0.16666666666666666, 0.0, 0.8333333333333334]","[0.16666666666666666, 0.0, 0.16666666666666666...","[0.0, 0.0, 0.11111111, 0.0, 0.0, 0.16666667, 0..."
3,HOY QUIERO FELICITAR A TODAS LAS MUJERES DE ES...,110593.jpeg,memes/110593.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Hispano or Latino, White ...","[High school degree or equivalent, Master’s de...","[Poland, Portugal, Spain, United Kingdom, Mexi...","{'users': ['ES1', 'ES3'], 'modalities': {'ET':...","[NO, YES, NO, NO, NO, NO]","[NONE, JUDGEMENTAL, NONE, NONE, NONE, NONE]","[[NONE], [STEREOTYPING-DOMINANCE, OBJECTIFICAT...",hoy quiero felicitar a todas las mujeres de es...,"[0.16666666666666666, 0.8333333333333334]",110593,"[0.8333333333333334, 0.16666666666666666, 0.0]","[0.8333333333333334, 0.0, 0.0, 0.0, 0.16666666...","[0.0, 0.0, 0.16666667, 0.0, 0.0, 0.16666667, 0..."
4,DUCHATE GUARRA GRACIAS memegenerator.es,110946.jpeg,memes/110946.jpeg,"[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Hispano or Latino, White ...","[Bachelor’s degree, Bachelor’s degree, Bachelo...","[Portugal, Chile, Italy, Portugal, Spain, Chile]","{'users': ['ES1', 'ES2', 'ES3', 'ES4'], 'modal...","[YES, YES, YES, YES, NO, NO]","[DIRECT, DIRECT, DIRECT, DIRECT, NONE, NONE]","[[OBJECTIFICATION], [SEXUAL-VIOLENCE], [OBJECT...",duchate guarra gracias memegenerator es,"[0.6666666666666666, 0.3333333333333333]",110946,"[0.3333333333333333, 0.0, 0.6666666666666666]","[0.3333333333333333, 0.0, 0.0, 0.1666666666666...","[0.0, 0.0, 0.11111111, 0.0, 0.0, 0.22222222, 0..."


## 10. O "Dataset" do PyTorch



Um `Dataset` do PyTorch é só um objeto que sabe responder duas perguntas: "quantas amostras

existem?" (`__len__`) e "me dá a amostra número N, já pronta pro modelo" (`__getitem__`).

Aqui, "pronta pro modelo" significa: texto tokenizado, imagem processada pelo CLIP, vetor

demográfico, e os 3 soft labels — tudo como tensores do PyTorch.



**Casos especiais tratados abaixo:**

- **Imagem ausente/corrompida** → uma imagem preta 224×224 é usada no lugar (não quebra o

  treino, mas o sinal daquela amostra fica "vazio" para o CLIP).

- **Meme sem voto válido na Task B ou C** → em vez de um soft label de verdade, colocamos um

  vetor "sentinela" preenchido com `-1.0`. Isso é um sinalizador especial (não é uma

  distribuição de probabilidade real!) que o modelo detecta mais adiante e usa para **ignorar

  essa amostra** no cálculo daquela loss específica, sem descartar a amostra inteira (as

  outras tasks daquela amostra continuam valendo).



Renomeei `self.labels_task1/2/3` (nomes do notebook original) para `self.labels_a/b/c` —

são exatamente a Task A/B/C, e o nome antigo (`task1`/`task2`/`task3`) confundia com os

nomes de coluna do dataset (`labels_task2_1`, `labels_task2_2`...), que vêm do formato

oficial do EXIST e não podem ser renomeados.


In [23]:
class SexismDataset(Dataset):
    def __init__(self, dataframe, tokenizer, clip_processor, max_length, image_dir):
        self.tokenizer      = tokenizer
        self.clip_processor = clip_processor
        self.max_length     = max_length
        self.image_dir      = image_dir

        self.text        = dataframe['text_preprocessed'].values
        self.image_paths = dataframe['path_memes'].values
        self.labels_a    = dataframe['labels_task2_1_soft'].values
        self.labels_b    = dataframe['labels_task2_2_soft'].values
        self.labels_c    = dataframe['labels_task2_3_soft'].values
        self.demo_vecs   = dataframe['demo_vec'].values

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):
        tokenized = self.tokenizer(
            self.text[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        image_path = os.path.join(self.image_dir, self.image_paths[idx])
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception:
            print("Imagem não encontrada:", image_path)
            image = Image.new('RGB', (224, 224), color=0)

        image_inputs = self.clip_processor(images=image, return_tensors='pt')

        batch = {
            'input_ids':      tokenized['input_ids'].squeeze(0),
            'attention_mask': tokenized['attention_mask'].squeeze(0),
            'pixel_values':   image_inputs['pixel_values'].squeeze(0),
            'demo_vec':       torch.tensor(self.demo_vecs[idx], dtype=torch.float32),
            'labels_a':       torch.tensor(self.labels_a[idx], dtype=torch.float32),
        }

        label_b = self.labels_b[idx]
        if label_b is None:
            batch['labels_b'] = torch.full((len(LABELS_2_2),), -1.0, dtype=torch.float32)
        else:
            label_b = np.array(label_b, dtype=np.float32)
            if label_b[0] != -1.0:
                batch['labels_b'] = torch.tensor(label_b, dtype=torch.float32)
            else:
                batch['labels_b'] = torch.full((len(label_b),), -1.0, dtype=torch.float32)

        label_c = self.labels_c[idx]
        if label_c is None:
            batch['labels_c'] = torch.full((len(LABELS_2_3),), -1.0, dtype=torch.float32)
        else:
            label_c = np.array(label_c, dtype=np.float32)
            if label_c[0] != -1.0:
                batch['labels_c'] = torch.tensor(label_c, dtype=torch.float32)
            else:
                batch['labels_c'] = torch.full((len(label_c),), -1.0, dtype=torch.float32)

        return batch

## 11. A arquitetura do modelo (`MultiHeadModel`)



O modelo tem 3 "entradas" (texto, imagem, demografia dos anotadores) que se combinam num

único vetor, e 3 "cabeças" de saída (uma por subtask). Em alto nível:



```

texto  ──[XLM-RoBERTa-large]──> CLS token (1024-d) ──[projeta p/ 512-d]──┐

                                                                          │

demografia (66-d) ──[gate + projeção]────────────────> soma no texto ───┤

                                                                          ├──[concatena]──> 1024-d ──> 3 cabeças ──> 3 previsões

imagem ──[CLIP ViT-L/14]──> CLS token (1024-d) ──[projeta p/ 512-d]──────┘

```



**Por que concatenar em vez de usar cross-attention (texto "olhando" para a imagem e

vice-versa)?** Cross-attention foi testada e descartada durante o desenvolvimento: com um

dataset pequeno (~5.000 memes), ela colapsava para padrões quase uniformes (o mecanismo de

atenção não conseguia aprender nada útil e virava só ruído). A concatenação simples, mais

"burra", generalizou melhor.



**Por que a demografia entra por "soma com portão" (gate) e não por concatenação?** A

concatenação de um vetor pequeno (66-d) direto num vetor grande (1024-d) faz o sinal

demográfico ficar "diluído" — o modelo tende a ignorá-lo. Somar ao vetor de texto, com um

portão que o próprio modelo aprende a abrir ou fechar por amostra, deu resultado melhor:

`texto_final = texto + portão × demografia_projetada`, onde `portão = sigmoid(...)` é um

número entre 0 e 1 por posição do vetor — funciona como um controle de volume que o modelo

ajusta sozinho durante o treino.



**Por que só as últimas 4 camadas do CLIP ficam treináveis?** O CLIP já vem pré-treinado

com bastante conhecimento visual geral. Treinar o modelo inteiro num dataset de 5.000

imagens arriscaria "esquecer" esse conhecimento (overfitting) e custaria muito mais memória.

Só descongelamos as últimas 4 camadas, que são as mais específicas para a tarefa.



**As 3 cabeças de saída** usam funções de erro diferentes porque as perguntas são diferentes:

- **Task A** (sexista? sim/não) → `BCEWithLogitsLoss`, pergunta binária.

- **Task B** (qual intenção?) → softmax + cross-entropy contra o soft label, porque as 3

  opções (NONE/JUDGEMENTAL/DIRECT) são mutuamente exclusivas — só uma é "a resposta".

- **Task C** (quais categorias?) → `BCEWithLogitsLoss` por categoria, porque mais de uma

  categoria pode estar certa ao mesmo tempo (não são mutuamente exclusivas).



**Flags novas (`use_vision`, `use_demographics`):** o notebook original já tinha esses

comportamentos como opcionais implicitamente (`if pixel_values is not None`, `if demo_vec is

not None`). Só tornei isso explícito na assinatura do construtor — não muda nenhum

comportamento com os valores padrão (`True`, `True`), mas deixa claro, olhando só a

assinatura, que essas duas entradas são desligáveis. Isso também prepara o terreno para o

estudo de ablação (rodar o modelo só-com-texto, só-com-texto+imagem, etc.) sem precisar

mexer no código do modelo depois — **os valores padrão continuam idênticos ao notebook

original**, então isto não é uma mudança de comportamento, só uma mudança de interface.


In [24]:
class MultiHeadModel(nn.Module):
    def __init__(self, num_labels_task_b, num_labels_task_c, demo_vec_dim=66,
                 dropout_prob=0.1, use_vision=True, use_demographics=True):
        super().__init__()
        self.use_vision = use_vision
        self.use_demographics = use_demographics

        self.encoder = XLMRobertaModel.from_pretrained("xlm-roberta-large")
        text_dim = self.encoder.config.hidden_size

        shared_dim = 512
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, shared_dim), nn.LayerNorm(shared_dim)
        )

        if self.use_vision:
            self.vision_encoder = CLIPVisionModel.from_pretrained(
                "openai/clip-vit-large-patch14"
            )
            vision_dim = self.vision_encoder.config.hidden_size
            frozen_layers = 4
            for param in self.vision_encoder.parameters():
                param.requires_grad = False
            for layer in self.vision_encoder.encoder.layers[-frozen_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True
            self.vision_proj = nn.Sequential(
                nn.Linear(vision_dim, shared_dim), nn.LayerNorm(shared_dim)
            )

        if self.use_demographics:
            self.demo_gate = nn.Linear(demo_vec_dim, shared_dim)
            self.demo_proj = nn.Sequential(
                nn.Linear(demo_vec_dim, shared_dim),
                nn.LayerNorm(shared_dim),
            )

        fused_dim = shared_dim * 2
        self.fusion_norm = nn.LayerNorm(fused_dim)
        self.dropout = nn.Dropout(dropout_prob)

        self.head_a = nn.Linear(fused_dim, 1)
        self.head_b = nn.Linear(fused_dim, num_labels_task_b)
        self.head_c = nn.Linear(fused_dim, num_labels_task_c)

        self.loss_a_fn = nn.BCEWithLogitsLoss()
        self.loss_c_fn = nn.BCEWithLogitsLoss(reduction='none')

    def _loss_task_a(self, logits_a, labels_a):
        if labels_a is None:
            return None
        if labels_a.dim() == 2:
            labels_a = labels_a[:, 1]
        mask_a = labels_a != -100.0
        if not mask_a.any():
            return None
        loss_a = self.loss_a_fn(logits_a[mask_a].view(-1), labels_a[mask_a])
        return loss_a * 2.0

    def _loss_task_b(self, logits_b, labels_b):
        if labels_b is None:
            return None
        mask_b = labels_b[:, 0] != -1.0
        if not mask_b.any():
            return None
        log_probs_b = F.log_softmax(logits_b[mask_b], dim=-1)
        return -(labels_b[mask_b] * log_probs_b).sum(dim=-1).mean()

    def _loss_task_c(self, logits_c, labels_c):
        if labels_c is None:
            return None
        mask_c = labels_c[:, 0] != -1.0
        if not mask_c.any():
            return None
        return self.loss_c_fn(logits_c[mask_c], labels_c[mask_c]).mean()

    def forward(self, input_ids, attention_mask,
                pixel_values=None, demo_vec=None,
                labels_a=None, labels_b=None, labels_c=None, **kwargs):

        text_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_cls = self.text_proj(text_out.last_hidden_state[:, 0, :])

        if self.use_demographics and demo_vec is not None:
            gate = torch.sigmoid(self.demo_gate(demo_vec))
            text_cls = text_cls + gate * self.demo_proj(demo_vec)

        if self.use_vision and pixel_values is not None:
            vision_out = self.vision_encoder(pixel_values=pixel_values)
            vision_cls = self.vision_proj(vision_out.last_hidden_state[:, 0, :])
        else:
            vision_cls = torch.zeros_like(text_cls)

        fused = torch.cat([text_cls, vision_cls], dim=-1)
        fused = self.fusion_norm(fused)
        fused = self.dropout(fused)

        logits_a = self.head_a(fused)
        logits_b = self.head_b(fused)
        logits_c = self.head_c(fused)

        task_losses = [
            self._loss_task_a(logits_a, labels_a),
            self._loss_task_b(logits_b, labels_b),
            self._loss_task_c(logits_c, labels_c),
        ]
        task_losses = [l for l in task_losses if l is not None]

        if task_losses:
            loss = sum(task_losses)
        else:
            loss = logits_a.sum() * 0.0

        return (loss, logits_a, logits_b, logits_c)

### 11.1 Instanciar o modelo e contar parâmetros


In [25]:
set_full_determinism(SEED)

# O `MultiHeadModel` é construído EXACTAMENTE como no A0 -- mesma ordem de draws
# de RNG -- e só depois é embrulhado. A camada de confusão nasce de constantes,
# não consome RNG: o fingerprint do modelo-base tem de bater com o do A0.
base_model = MultiHeadModel(
    num_labels_task_b=len(LABELS_2_2),
    num_labels_task_c=len(LABELS_2_3),
    demo_vec_dim=DEMO_VEC_DIM,
)
fp_base_init = model_fingerprint(base_model)
print("Fingerprint dos pesos iniciais (base, comparável com o A0):", fp_base_init)

vote_table, ANNOTATOR_IDS, vote_stats = build_vote_table(
    './Dataset/memes/training/EXIST2026_training.json'
)

model = ConfusionMultiHeadModel(
    base_model,
    n_annotators=len(ANNOTATOR_IDS),
    method=METHOD,
    lam=LAMBDA,
    reg_weight=REG_WEIGHT,
)
print(f"método={METHOD} lambda={LAMBDA} reg={model.confusion.reg_weight} "
      f"anotadores={len(ANNOTATOR_IDS)}")
print("Fingerprint dos pesos iniciais (com confusão):", model_fingerprint(model))


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.embeddings.position_embedding.

Fingerprint dos pesos iniciais (base, comparável com o A0): 6cd0ef763f16d975
[confusion] itens=3984 anotadores=887 votos=23904 descartados=0 votos/anotador=26.95
método=M1 lambda=0.0 reg=0.01 anotadores=887
Fingerprint dos pesos iniciais (com confusão): f0778b67ca366c0d


In [26]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
frozen_params = total_params - trainable_params

print(f"Parâmetros totais:     {total_params:,}")
print(f"Parâmetros treináveis: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
print(f"Parâmetros congelados: {frozen_params:,}")

Parâmetros totais:     864,207,338
Parâmetros treináveis: 611,412,458 (70.7%)
Parâmetros congelados: 252,794,880


In [27]:
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-large")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

def majority_vote_task_a(soft_label):
    return 'YES' if soft_label[0] >= 0.5 else 'NO'

with open('../reliability/outputs/split_taskA_frozen.json') as f:
    _SPLIT = json.load(f)

_by_id = dataset.set_index('_id_frozen', drop=False)
dataset_train = _by_id.loc[_SPLIT['train_ids']].reset_index(drop=True)
dataset_val   = _by_id.loc[_SPLIT['val_ids']].reset_index(drop=True)

with open('../reliability/outputs/soft_labels_taskA_raw.json') as f:
    _SOFT_A_RAW = json.load(f)
dataset_val['labels_task2_1_soft'] = dataset_val['_id_frozen'].map(_SOFT_A_RAW)

IMAGE_DIR = './Dataset/memes/training/'

train_dataset = SexismDataset(
    dataframe=dataset_train,
    tokenizer=tokenizer,
    clip_processor=clip_processor,
    max_length=128,
    image_dir=IMAGE_DIR,
)

val_dataset = SexismDataset(
    dataframe=dataset_val,
    tokenizer=tokenizer,
    clip_processor=clip_processor,
    max_length=128,
    image_dir=IMAGE_DIR,
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False)

# Só o TREINO recebe os votos por anotador. A validação fica crua, logo
# `eval_loss` continua a ser exactamente a L_soft do A0 -- e E = argmin(eval_loss)
# é o mesmo selector de época nos dois lados da comparação.
train_dataset_aux = VoteAugmentedDataset(
    train_dataset, dataset_train['_id_frozen'].tolist(), vote_table
)

print(f"Treino: {len(train_dataset)} amostras | Validação: {len(val_dataset)} amostras")
print(f"Votos anexados ao treino: {len(train_dataset_aux)} itens x 6 votos")

Treino: 3386 amostras | Validação: 598 amostras
Votos anexados ao treino: 3386 itens x 6 votos


## 13. Configuração de treino



Cada parâmetro abaixo controla um aspecto do processo de treino:



| Parâmetro | O que faz |

|---|---|

| `num_train_epochs` | Quantas vezes o modelo passa pelo dataset de treino inteiro. |

| `per_device_train/eval_batch_size` | Quantas amostras processadas juntas em cada passo. |

| `gradient_accumulation_steps` | Acumula gradiente de vários batches pequenos antes de atualizar os pesos — simula um batch maior (8×8=64) sem precisar de mais memória de GPU de uma vez. |

| `learning_rate` | O tamanho do "passo" que o modelo dá a cada atualização de pesos. |

| `warmup_steps` | Início do treino com taxa de aprendizado crescendo aos poucos (em vez de já começar no valor máximo), para não desestabilizar os pesos pré-treinados logo de cara. |

| `weight_decay` | Penaliza pesos muito grandes, ajudando a evitar overfitting. |

| `eval_strategy` / `save_strategy` | Quando avaliar/salvar um checkpoint — aqui, a cada época. |

| `load_best_model_at_end` | Ao final do treino, recarrega o checkpoint que teve o melhor resultado na validação (não necessariamente o último). |

| `metric_for_best_model` | Qual métrica decide "melhor" — aqui, `eval_loss` (regra ARCH-06: escolher a época por argmin da loss, nunca pela métrica reportada). |

| `fp16` | Usa números de 16 bits em vez de 32 bits em partes do treino — mais rápido e usa menos memória, com perda de precisão numérica desprezível na prática. |

| `seed` / `data_seed` | Sementes dos geradores usados pelo Trainer (dropout, embaralhamento dos batches). Explícitas por reprodutibilidade — ver seção 2.1. |


In [28]:
training_args = TrainingArguments(
    output_dir="./results",
    seed=SEED,
    data_seed=SEED,
    num_train_epochs=15,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=0.1,
    weight_decay=0.05,
    # --- SO PARA O TESTE DE 2 PASSOS: nao usar noutras runs ---
    max_steps=2,
    logging_steps=1,
    eval_strategy="no",
    save_strategy="no",
    load_best_model_at_end=False,
    # -----------------------------------------------------------
    save_total_limit=2,
    save_only_model=True,
    fp16=True,
    report_to="none",
    dataloader_pin_memory=True,
)

## 14. Métricas de avaliação (PyEvALL)



A competição EXIST usa a biblioteca oficial **PyEvALL** para calcular a métrica **ICM-Norm**

(Information Contrast Model, normalizada), em duas variantes:

- **soft** — compara a distribuição de probabilidade inteira prevista contra a distribuição

  real de votos dos anotadores (penaliza menos um erro "quase certo").

- **hard** — compara um rótulo único previsto contra o rótulo único do gabarito (para chegar

  num rótulo único a partir de uma distribuição, é preciso um **threshold**, ver seção 14.2).



A função `compute_metrics` roda a cada época de validação e devolve um dicionário de métricas

que o `Trainer` usa para decidir qual checkpoint é o melhor (`metric_for_best_model` acima).


### 14.1 Funções auxiliares — formatar e chamar o PyEvALL



Estas funções convertem as previsões do modelo para o formato JSON que o PyEvALL espera, e

extraem os números de volta do relatório que ele devolve. A lógica interna é copiada

literalmente do notebook original (é a parte mais delicada do pipeline — qualquer mudança de

comportamento aqui muda os números reportados), com comentários adicionados para explicar

cada bloco.


In [29]:
def _extract_metrics_from_report(report, metric_list):
    results = {}
    if not hasattr(report, 'report') or 'metrics' not in report.report:
        return results

    metrics_dict = report.report['metrics']
    for metric in metric_list:
        if metric not in metrics_dict:
            continue
        if metrics_dict[metric].get('status') != 'OK':
            continue
        try:
            results[metric] = float(
                metrics_dict[metric]['results']['average_per_test_case']
            )
        except (KeyError, TypeError, ValueError):
            pass
    return results

In [30]:
def _format_for_pyevall(probs, sample_ids, task, evaluation, thresholds_c=None):
    out = []
    for i, sid in enumerate(sample_ids):
        entry = {"test_case": "EXIST2025", "id": str(sid)}

        if task == '2_1':
            p_yes = float(probs[i][0])
            p_no  = float(probs[i][1])
            if evaluation == 'hard':
                entry["value"] = "YES" if p_yes >= p_no else "NO"
            else:
                entry["value"] = {"YES": p_yes, "NO": p_no}

        elif task == '2_2':
            label_map = {"NONE": "NO", "JUDGEMENTAL": "JUDGEMENTAL", "DIRECT": "DIRECT"}
            relevant  = {label_map[lbl]: float(probs[i][idx])
                         for idx, lbl in enumerate(LABELS_2_2)}
            if evaluation == 'hard':
                entry["value"] = max(relevant, key=relevant.get)
            else:
                entry["value"] = relevant

        elif task == '2_3':
            label_map = {"NONE": "NO"}
            label_map.update({lbl: lbl for lbl in LABELS_2_3[1:]})
            relevant  = {label_map[lbl]: float(probs[i][idx])
                         for idx, lbl in enumerate(LABELS_2_3)}

            if evaluation == 'hard':
                threshs = thresholds_c if thresholds_c is not None else [0.5] * len(LABELS_2_3)
                thresh_map = {label_map[lbl]: threshs[idx] for idx, lbl in enumerate(LABELS_2_3)}
                hard_labels = [lbl for lbl, p in relevant.items() if p >= thresh_map[lbl]]
                if not hard_labels:
                    hard_labels = [max(relevant, key=relevant.get)]
                entry["value"] = hard_labels
            else:
                entry["value"] = relevant

        out.append(entry)
    return out

In [31]:
def _run_pyevall(probs, gold_dist, sample_ids, task, evaluation,
                 temp_dir='./pyevall_temp_train', thresholds_c=None):
    os.makedirs(temp_dir, exist_ok=True)

    preds = _format_for_pyevall(probs, sample_ids, task, evaluation,
                                 thresholds_c=thresholds_c)
    gold  = _format_for_pyevall(gold_dist, sample_ids, task, evaluation)

    pred_path = f'{temp_dir}/pred_{task}_{evaluation}.json'
    gold_path = f'{temp_dir}/gold_{task}_{evaluation}.json'
    with open(pred_path, 'w') as f: json.dump(preds, f)
    with open(gold_path, 'w') as f: json.dump(gold, f)

    if evaluation == 'hard':
        metric_list = ['ICM', 'ICMNorm', 'FMeasure']
    else:
        if task == '2_3':
            metric_list = ['ICMSoft', 'ICMSoftNorm']
        else:
            metric_list = ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']

    params = {}
    if task == '2_2':
        params[PyEvALLUtils.PARAM_HIERARCHY] = TASK_2_2_HIERARCHY
    elif task == '2_3':
        params[PyEvALLUtils.PARAM_HIERARCHY] = TASK_2_3_HIERARCHY

    try:
        evaluator = PyEvALLEvaluation()
        report    = evaluator.evaluate(pred_path, gold_path, metric_list, **params)
        return _extract_metrics_from_report(report, metric_list)
    except Exception as e:
        print(f"  ! PyEvALL falhou para task {task} ({evaluation}): {e}")
        return {}

### 14.2 Busca do melhor threshold da Task C, em português claro



A Task C prevê 6 categorias independentes, cada uma com uma probabilidade entre 0 e 1. Para

virar uma decisão "está presente / não está" (modo *hard*), é preciso um valor de corte —

o **threshold**. Usar 0.5 para todas é a opção óbvia, mas nem sempre é a melhor: uma

categoria rara pode se beneficiar de um corte mais baixo (ex.: 0.30), outra mais comum de um

corte mais alto.



**Como a busca funciona, passo a passo** (chamada de "busca gulosa por coordenada"):

1. Começa com threshold 0.5 para as 6 categorias.

2. Pega a categoria 1 (`NONE`). Testa 12 valores de corte diferentes para ela (0.05, 0.10,

   ..., 0.60), **mantendo as outras 5 categorias fixas em 0.5**. Fica com o valor que deu o

   melhor ICM-Norm hard.

3. Repete o mesmo processo para a categoria 2, depois a 3, e assim por diante — cada categoria

   "herda" o melhor valor já encontrado para as anteriores.

4. No final, tem um threshold ajustado para cada uma das 6 categorias.



**Custo:** 12 valores testados × 6 categorias = 72 chamadas ao PyEvALL, **a cada época de

validação** (rodou 30 vezes durante o treino inteiro). Esse ajuste, quando aplicado, melhorou

o ICM hard da Task C de 0,3761 para 0,3948 (ver `CLAUDE.md` do repositório).


In [32]:
def _search_thresholds_task_c(probs_c, targets_c, ids_c, temp_dir):
    best_thresholds = [0.5] * len(LABELS_2_3)

    for class_idx in range(len(LABELS_2_3)):
        best_t   = best_thresholds[class_idx]
        best_icm = -np.inf

        for t in THRESHOLD_GRID:
            candidate = best_thresholds.copy()
            candidate[class_idx] = float(t)

            scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                  temp_dir=temp_dir, thresholds_c=candidate)
            icm = scores.get('ICMNorm', -np.inf)
            if icm > best_icm:
                best_icm = icm
                best_t   = float(t)

        best_thresholds[class_idx] = best_t

    final_scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                temp_dir=temp_dir, thresholds_c=best_thresholds)
    return best_thresholds, final_scores

### 14.3 `compute_metrics` — a função principal chamada pelo `Trainer`



Roda 6 avaliações (3 tasks × {soft, hard}) mais a busca de threshold da Task C, a cada época.

Convenção importante para ler o bloco da Task A: o modelo foi treinado com a loss calculada

contra `labels_a[:, 1]`, que é **P(NO)** (`LABELS_2_1 = ['YES', 'NO']`, índice 1 = NO) —

então `sigmoid(logit)` dá `P(NO)` diretamente, e `P(YES) = 1 - P(NO)`. É por isso que

`prob_no` (não `prob_yes`) é a variável calculada primeiro no bloco abaixo.


In [33]:
def _metrics_task_a(logits_a, labels_a, temp_dir):
    metrics = {}
    if labels_a.ndim == 2:
        labels_a = labels_a[:, 1]
    mask_a = labels_a != -100.0
    if mask_a.any():
        prob_no  = 1 / (1 + np.exp(-logits_a[mask_a].squeeze(-1)))
        probs_a  = np.stack([1 - prob_no, prob_no], axis=-1)
        targets_a = np.stack([1 - labels_a[mask_a], labels_a[mask_a]], axis=-1)
        ids_a    = [str(i) for i in range(len(probs_a))]

        hard_scores = _run_pyevall(probs_a, targets_a, ids_a, '2_1', 'hard',
                                   temp_dir=temp_dir)
        for k, v in hard_scores.items():
            metrics[f'task_a_{k.lower()}_hard'] = v

        soft_scores = _run_pyevall(probs_a, targets_a, ids_a, '2_1', 'soft',
                                   temp_dir=temp_dir)
        for k, v in soft_scores.items():
            metrics[f'task_a_{k.lower()}_soft'] = v

    return metrics


def _metrics_task_b(logits_b, labels_b, temp_dir):
    metrics = {}
    mask_b = labels_b[:, 0] != -1.0
    if mask_b.any():
        x        = logits_b[mask_b]
        x        = x - x.max(axis=-1, keepdims=True)
        probs_b  = np.exp(x) / np.exp(x).sum(axis=-1, keepdims=True)
        targets_b = labels_b[mask_b]
        ids_b    = [str(i) for i in range(len(probs_b))]

        hard_scores = _run_pyevall(probs_b, targets_b, ids_b, '2_2', 'hard',
                                   temp_dir=temp_dir)
        for k, v in hard_scores.items():
            metrics[f'task_b_{k.lower()}_hard'] = v

        soft_scores = _run_pyevall(probs_b, targets_b, ids_b, '2_2', 'soft',
                                   temp_dir=temp_dir)
        for k, v in soft_scores.items():
            metrics[f'task_b_{k.lower()}_soft'] = v

    return metrics


def _metrics_task_c(logits_c, labels_c, temp_dir):
    metrics = {}
    mask_c = labels_c[:, 0] != -1.0
    if mask_c.any():
        probs_c  = 1 / (1 + np.exp(-logits_c[mask_c]))
        targets_c = labels_c[mask_c]
        ids_c    = [str(i) for i in range(len(probs_c))]

        soft_scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'soft',
                                   temp_dir=temp_dir)
        for k, v in soft_scores.items():
            metrics[f'task_c_{k.lower()}_soft'] = v

        hard_scores_default = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                           temp_dir=temp_dir)
        for k, v in hard_scores_default.items():
            metrics[f'task_c_{k.lower()}_hard_default'] = v

        best_thresholds, hard_scores_tuned = _search_thresholds_task_c(
            probs_c, targets_c, ids_c, temp_dir
        )
        for k, v in hard_scores_tuned.items():
            metrics[f'task_c_{k.lower()}_hard'] = v

        for j, label in enumerate(LABELS_2_3):
            metrics[f'task_c_thresh_{label.lower().replace("-", "_")}'] = round(best_thresholds[j], 2)

    return metrics


def compute_metrics(eval_pred):
    logits_a, logits_b, logits_c = eval_pred.predictions
    labels_a, labels_b, labels_c = eval_pred.label_ids
    logits_a = np.asarray(logits_a)
    logits_b = np.asarray(logits_b)
    logits_c = np.asarray(logits_c)
    labels_a = np.asarray(labels_a)
    labels_b = np.asarray(labels_b)
    labels_c = np.asarray(labels_c)
    metrics = {}

    with tempfile.TemporaryDirectory() as temp_dir:
        metrics.update(_metrics_task_a(logits_a, labels_a, temp_dir))
        metrics.update(_metrics_task_b(logits_b, labels_b, temp_dir))
        metrics.update(_metrics_task_c(logits_c, labels_c, temp_dir))

    return metrics

## 15. Treino



Instancia o `Trainer` do HuggingFace com o modelo, os argumentos de treino, os datasets e a

função de métricas definidos acima, e roda `trainer.train()`.


### 15.1 Salvar métricas de cada época (CSV + JSON)



O `Trainer` já acumula um histórico de tudo que loga (`trainer.state.log_history`) — uma

lista de dicionários, um por evento de log/avaliação, com chaves diferentes dependendo do

tipo (passo de treino vs. avaliação de época). O callback abaixo salva esse histórico em

CSV e JSON **a cada época**, sobrescrevendo o mesmo par de arquivos — assim, se o treino for

interrompido na VM, o histórico até a última época concluída não se perde.



O nome do arquivo usa o **timestamp de início da run** (`YYYYMMDD_HHMMSS`), gerado uma vez

quando o callback é criado, para que todas as épocas de uma mesma run gravem no mesmo par de

arquivos, e runs diferentes nunca se sobrescrevam.


In [34]:
class MetricsHistoryLogger(TrainerCallback):
    def __init__(self, output_dir="./metrics_history"):
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)
        self.run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S") + f"_{RUN_TAG}"
        self.json_path = os.path.join(self.output_dir, f"{self.run_timestamp}.json")
        self.csv_path = os.path.join(self.output_dir, f"{self.run_timestamp}.csv")

    def _save(self, log_history):
        with open(self.json_path, "w", encoding="utf-8") as f:
            json.dump(log_history, f, indent=2, ensure_ascii=False)

        all_keys = []
        for row in log_history:
            for key in row:
                if key not in all_keys:
                    all_keys.append(key)

        with open(self.csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=all_keys)
            writer.writeheader()
            writer.writerows(log_history)

    def on_evaluate(self, args, state, control, **kwargs):
        self._save(state.log_history)

    def on_train_end(self, args, state, control, **kwargs):
        self._save(state.log_history)

In [35]:
metrics_logger = MetricsHistoryLogger()
ConfusionMatrixLogger = make_confusion_logger_class(TrainerCallback)
confusion_logger = ConfusionMatrixLogger(
    model, ANNOTATOR_IDS,
    metrics_logger.json_path.replace(".json", "_confusion.json"),
)
print(f"Métricas: {metrics_logger.json_path}")
print(f"Matrizes: {confusion_logger.out_path}")


Métricas: ./metrics_history/20260907_083944_conf_M1_lam0.0_seed42.json
Matrizes: ./metrics_history/20260907_083944_conf_M1_lam0.0_seed42_confusion.json


In [36]:
# `ConfusionTrainer` exclui `confusion.*` do weight decay (0.05): decaimento nos
# logits das matrizes puxaria para a matriz UNIFORME, o oposto de "perto da
# identidade" -- e no M3 seria uma regularização que a especificação proíbe.
ConfusionTrainer = make_confusion_trainer_class(Trainer)

trainer = ConfusionTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_aux,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[metrics_logger, confusion_logger],
)

decayed = trainer.get_decay_parameter_names(model)
assert not [n for n in decayed if n.startswith("confusion.")], "confusão no weight decay"
print("OK: parâmetros de confusão fora do weight decay")


OK: parâmetros de confusão fora do weight decay


In [37]:
trainer.train()

Step,Training Loss
1,3.473207
2,3.403258


TrainOutput(global_step=2, training_loss=3.4382323026657104, metrics={'train_runtime': 5.2233, 'train_samples_per_second': 24.505, 'train_steps_per_second': 0.383, 'total_flos': 0.0, 'train_loss': 3.4382323026657104, 'epoch': 0.03773584905660377})

In [38]:
for r in trainer.state.log_history:
    if "loss" in r and "eval_loss" not in r:
        print(f"step {r['step']}  loss {r['loss']:.10f}  grad_norm {r['grad_norm']:.10f}")

step 1  loss 3.4732069969  grad_norm 15.6839170456
step 2  loss 3.4032576084  grad_norm inf
